# Cinemática Inversa Numérica

In [1]:
import numpy as np
from numpy import float32, float64, int32, int64
from dataclasses import dataclass
from typing import Self

## Versión inicial inexacta

In [2]:
@dataclass
class Pose:
  rotacion: np.ndarray
  translacion: np.ndarray

  @classmethod
  def identidad(cls) -> Self:
    return Pose(
      np.eye(3),
      np.zeros(3),
    )

  def __mul__(self, pose: Self) -> Self:
    return Pose(
      self.rotacion @ pose.rotacion,
      self.rotacion @ pose.translacion + self.translacion,
    )

  def distancia_cuadrado(self, pose: Self) -> float64:
    diff = self.translacion - pose.translacion
    return diff[0]**2 + diff[1]**2 + diff[2]**2

def DH(theta: float64, d: float64, a: float64, alpha: float64) -> Pose:
    cth = np.cos(theta)
    sth = np.sin(theta)
    ca = np.cos(alpha)
    sa = np.sin(alpha)

    return Pose(np.array([
        [cth, -sth*ca,  sth*sa],
        [sth,  cth*ca, -cth*sa],
        [  0,     sa,       ca],
    ]), np.array([ a*cth, a*sth, d ]))

In [3]:
@dataclass
class Dimensiones:
  d1: float64
  a1: float64
  a2: float64
  d4: float64
  d6: float64

@dataclass
class Configuracion:
  c1: bool
  c2: bool
  c3: bool

class IRB140:
  def __init__(self, dim: Dimensiones, ABase0: Pose | None = None, A6Tool: Pose | None = None):
    self.dim = dim

    self.A01 = lambda q1: DH(q1, dim.d1, dim.a1, -np.pi/2)
    self.A12 = lambda q2: DH(q2,      0, dim.a2,        0)
    self.A23 = lambda q3: DH(q3,      0,      0,  np.pi/2)
    self.A34 = lambda q4: DH(q4, dim.d4,      0, -np.pi/2)
    self.A45 = lambda q5: DH(q5,      0,      0,  np.pi/2)
    self.A56 = lambda q6: DH(q6, dim.d6,      0,        0)

    self.ABase0 = Pose.identidad() if ABase0 is None else ABase0
    self.A6Tool = Pose.identidad() if A6Tool is None else A6Tool

  def cinematica_directa(self, q: np.ndarray) -> Pose:
    q1, q2, q3, q4, q5, q6 = tuple(q.tolist())
    A06 = self.A01(q1) * self.A12(q2) * self.A23(q3) * self.A34(q4) * self.A45(q5) * self.A56(q6)
    return self.ABase0 * A06 * self.A6Tool

  def configuracion(self, q: np.ndarray, epsilon: float64 = 1e-6) -> Configuracion | None:
    _, q2, q3, _, q5, _ = tuple(q.tolist())

    config1 = self.dim.d4 * np.sin(q2 + q3) + self.dim.a2 * np.cos(q2) + self.dim.a1
    config2 = np.cos(q3)
    config3 = q5 + 0.0001

    if abs(config1) < epsilon or abs(config2) < epsilon or abs(config3) < epsilon:
      return None

    return Configuracion(bool(config1 > 0), bool(config2 > 0), bool(config3 > 0))



In [4]:
def funcion_costo(pose_origen: Pose, pose_destino: Pose) -> np.float64:
  return pose_origen.distancia_cuadrado(pose_destino)

In [5]:
irb140 = IRB140(Dimensiones(0, 70, 360, 380, 0))

In [6]:
VARIACION = 1e-4

def cinematima_inversa(pose_destino: Pose, alfa: float64, epsilon: float64 = 1e-2, max_iteraciones: int32 = 1000) -> np.ndarray:
  q = np.array([0., 0., 0., 0., 0., 0.])

  for _ in range(max_iteraciones):
    costo = funcion_costo(irb140.cinematica_directa(q), pose_destino)
    if costo < epsilon:
      break

    gradiente = np.zeros(6)
    for i in range(6):
      delta_q = np.zeros(6)
      delta_q[i] += VARIACION

      delta_costo = funcion_costo(irb140.cinematica_directa(q + delta_q), pose_destino) - costo

      gradiente[i] = delta_costo / VARIACION

    q -= alfa * gradiente

  return q

In [7]:
CANT_PRUEBAS = 100
print(f"Numero de pruebas: {CANT_PRUEBAS}")

cant_acertada = 0
for _ in range(CANT_PRUEBAS):
  q_destino = 2 * np.pi * np.random.rand(6)
  q_destino[3:] *= 0
  q_obtenido = cinematima_inversa(irb140.cinematica_directa(q_destino), 1e-6, max_iteraciones = 1000)

  if np.linalg.norm(q_destino - q_obtenido) < 0.1:
    cant_acertada += 1

print(f"\tResultado: {cant_acertada}/{CANT_PRUEBAS}")

Numero de pruebas: 100
	Resultado: 1/100


In [8]:
VARIACION = 1e-4

def semilla(config: Configuracion) -> np.ndarray:
  match (config.c1, config.c2, config.c3):
    case (( True,  True,  True)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case (( True,  True, False)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case (( True, False,  True)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case (( True, False, False)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case ((False,  True,  True)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case ((False,  True, False)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case ((False, False,  True)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case ((False, False, False)): return np.array([ 0., 0., 0., 0., 0., 0., ])

def cinematima_inversa(pose_destino: Pose, config: Configuracion, alfa: float64, epsilon: float64 = 1e-2, max_iteraciones: int32 = 1000) -> np.ndarray:
  q = semilla(config)

  for _ in range(max_iteraciones):
    costo = funcion_costo(irb140.cinematica_directa(q), pose_destino)
    if costo < epsilon:
      break

    gradiente = np.zeros(6)
    for i in range(6):
      delta_q = np.zeros(6)
      delta_q[i] += VARIACION

      delta_costo = funcion_costo(irb140.cinematica_directa(q + delta_q), pose_destino) - costo

      gradiente[i] = delta_costo / VARIACION

    q -= alfa * gradiente

  return q

In [9]:
CANT_PRUEBAS = 100
print(f"Numero de pruebas con configuración: {CANT_PRUEBAS}")

cant_acertada = 0
for _ in range(CANT_PRUEBAS):
  q_destino = 2 * np.pi * np.random.rand(6)
  config = None

  while config is None:
    q_destino = 2 * np.pi * np.random.rand(6)
    q_destino[3:] *= 0
    config = irb140.configuracion(q_destino)

  q_obtenido = cinematima_inversa(irb140.cinematica_directa(q_destino), config, 1e-6, max_iteraciones = 1000)

  if np.linalg.norm(q_destino - q_obtenido) < 0.1:
    cant_acertada += 1

print(f"\tResultado: {cant_acertada}/{CANT_PRUEBAS}")

Numero de pruebas con configuración: 100
	Resultado: 5/100
